In [0]:

df = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")
df.printSchema()
display(df.limit(10))

print("Total de filas:", df.count())
print("Valores únicos en 'sex':")
df.select("sex").distinct().show()
print("Valores nulos por columna:")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql import functions as F

df = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")

print("Valores nulos por columna:")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql import functions as F

# Leer desde Bronce
df_bronze = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")

df_silver = (
    df_bronze
    
    # Regla 5 — correct_data_types: convertir a tipos correctos
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("number", F.col("number").cast("int"))  # es un conteo de muertes, debe ser entero
    .withColumn("percentage_of_cause_specific_deaths_out_of_total_deaths", 
                F.col("percentage_of_cause_specific_deaths_out_of_total_deaths").cast("double"))
    .withColumn("age_standardized_death_rate_per_100_000_standard_population", 
                F.col("age_standardized_death_rate_per_100_000_standard_population").cast("double"))
    .withColumn("death_rate_per_100_000_population", 
                F.col("death_rate_per_100_000_population").cast("double"))
    
    # Regla 6 — standardize_categorical_values: estandarizar sexo a código corto
    .withColumn("sex", 
                F.when(F.col("sex") == "Male", "M")
                 .when(F.col("sex") == "Female", "F")
                 .when(F.col("sex") == "All", "TODOS")
                 .otherwise(F.col("sex")))
    
    # Regla 6 — limpiar age_group: quitar corchetes de "[All]" -> "All"
    .withColumn("age_group", F.regexp_replace(F.col("age_group"), r"[\[\]]", ""))
    
    # Regla 7 — semantic_validation: eliminar filas con conteos negativos (imposible)
    .filter(F.col("number") >= 0)
    
    # Agregar país de origen (necesario para cuando consolidemos en Gold con otras fuentes)
    .withColumn("pais", F.lit("Costa Rica"))
    .withColumn("codigo_pais", F.lit("CRI"))
    
    # Metadata de control de Silver
    .withColumn("silver_processed_at", F.current_timestamp())
)

# Regla 3 — remove_duplicate_rows
filas_antes = df_silver.count()
df_silver = df_silver.dropDuplicates()
filas_despues = df_silver.count()
print(f"Filas antes de quitar duplicados: {filas_antes}")
print(f"Filas después de quitar duplicados: {filas_despues}")
print(f"Duplicados eliminados: {filas_antes - filas_despues}")

display(df_silver.limit(10))
df_silver.printSchema()

In [0]:
#Guardamos en esquema silver la primera tabla
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("covid19.silver.mortalidad_categorias_costa_rica_2020")

print("✓ Tabla Silver creada exitosamente")

# Verificar
spark.sql("SELECT COUNT(*) as total_filas FROM covid19.silver.mortalidad_categorias_costa_rica_2020").show()

In [0]:
tablas_a_revisar = [
    "mortalidad_categorias_costa_rica_2020",
    "mortalidad_categorias_costa_rica_2021", 
    "mortalidad_categorias_costa_rica_2022",
    "mortalidad_indicadores_costa_rica",
    "mortalidad_por_edades_costa_rica",
    "who_covid_19_global_daily_data"
]

for tabla in tablas_a_revisar:
    print(f"\n{'='*80}")
    print(f"TABLA: covid19.bronze.{tabla}")
    print(f"{'='*80}")
    df = spark.table(f"covid19.bronze.{tabla}")
    print(f"Número de columnas: {len(df.columns)}")
    print(f"Número de filas: {df.count()}")
    print("\nColumnas y tipos:")
    df.printSchema()
    print("\nPrimeras 3 filas:")
    display(df.limit(3))

In [0]:
# Ejecuten esto para ver si son datos duplicados o diferentes
print("=== AÑOS ÚNICOS en cada tabla ===")
print("categorias_2020/2021/2022:", 
      spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020").select("year").distinct().orderBy("year").collect())

print("indicadores:", 
      spark.table("covid19.bronze.mortalidad_indicadores_costa_rica").select("year").distinct().orderBy("year").collect())

print("por_edades:", 
      spark.table("covid19.bronze.mortalidad_por_edades_costa_rica").select("year").distinct().orderBy("year").collect())

print("\n=== INDICADORES ÚNICOS (primeros 10) ===")
print("categorias:", 
      spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020").select("indicator_code", "indicator_name").distinct().limit(10).collect())

print("indicadores:", 
      spark.table("covid19.bronze.mortalidad_indicadores_costa_rica").select("indicator_code", "indicator_name").distinct().limit(10).collect())

In [0]:
# Verificar si hay indicadores duplicados entre las tablas
df_categorias = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020").select("indicator_code").distinct()
df_indicadores = spark.table("covid19.bronze.mortalidad_indicadores_costa_rica").select("indicator_code").distinct()
df_edades = spark.table("covid19.bronze.mortalidad_por_edades_costa_rica").select("indicator_code").distinct()

print("Indicadores en categorias:", df_categorias.count())
print("Indicadores en indicadores:", df_indicadores.count())
print("Indicadores en por_edades:", df_edades.count())

# Verificar solapamiento
solapamiento_cat_ind = df_categorias.intersect(df_indicadores).count()
solapamiento_cat_edad = df_categorias.intersect(df_edades).count()
solapamiento_ind_edad = df_indicadores.intersect(df_edades).count()

print(f"\nSolapamiento categorias vs indicadores: {solapamiento_cat_ind}")
print(f"Solapamiento categorias vs por_edades: {solapamiento_cat_edad}")
print(f"Solapamiento indicadores vs por_edades: {solapamiento_ind_edad}")

In [0]:
from pyspark.sql import functions as F

# Leer TODAS las tablas de Costa Rica
df_cat_2020 = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")
df_cat_2021 = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2021")
df_cat_2022 = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2022")
df_indicadores = spark.table("covid19.bronze.mortalidad_indicadores_costa_rica")
df_edades = spark.table("covid19.bronze.mortalidad_por_edades_costa_rica")

# Unirlas TODAS
df_bronze_consolidado = (
    df_cat_2020
    .unionAll(df_cat_2021)
    .unionAll(df_cat_2022)
    .unionAll(df_indicadores)
    .unionAll(df_edades)
)

print(f"Filas totales consolidadas: {df_bronze_consolidado.count()}")

# Aplicar transformaciones Silver
df_silver = (
    df_bronze_consolidado
    
    # Regla 5 — correct_data_types
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("number", F.col("number").cast("int"))
    .withColumn("percentage_of_cause_specific_deaths_out_of_total_deaths", 
                F.col("percentage_of_cause_specific_deaths_out_of_total_deaths").cast("double"))
    .withColumn("age_standardized_death_rate_per_100_000_standard_population", 
                F.col("age_standardized_death_rate_per_100_000_standard_population").cast("double"))
    .withColumn("death_rate_per_100_000_population", 
                F.col("death_rate_per_100_000_population").cast("double"))
    
    # Regla 6 — standardize_categorical_values
    .withColumn("sex", 
                F.when(F.col("sex") == "Male", "M")
                 .when(F.col("sex") == "Female", "F")
                 .when(F.col("sex") == "All", "TODOS")
                 .otherwise(F.col("sex")))
    
    # Limpiar age_group: quitar corchetes
    .withColumn("age_group", F.regexp_replace(F.col("age_group"), r"[\[\]]", ""))
    
    # Regla 7 — semantic_validation
    .filter(F.col("number") >= 0)
    
    # Agregar metadata
    .withColumn("pais", F.lit("Costa Rica"))
    .withColumn("codigo_pais", F.lit("CRI"))
    .withColumn("fuente", F.lit("OMS/WHO - Costa Rica"))
    .withColumn("silver_processed_at", F.current_timestamp())
)

# Regla 3 — remove_duplicate_rows
filas_antes = df_silver.count()
df_silver = df_silver.dropDuplicates()
filas_despues = df_silver.count()
print(f"\nDuplicados eliminados: {filas_antes - filas_despues}")
print(f"Filas finales: {filas_despues}")

# Guardar en Silver (UNA SOLA TABLA)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("covid19.silver.mortalidad_costa_rica")

print("\n✓ Tabla Silver consolidada creada exitosamente: mortalidad_costa_rica")

# Verificar distribución por año
spark.sql("""
    SELECT year, COUNT(*) as filas 
    FROM covid19.silver.mortalidad_costa_rica 
    GROUP BY year 
    ORDER BY year
""").show()

In [0]:
# Eliminar la tabla anterior
spark.sql("DROP TABLE IF EXISTS covid19.silver.mortalidad_categorias_costa_rica_2020")
print("✓ Tabla individual anterior eliminada")

In [0]:
# Verificar la tabla consolidada
spark.sql("""
    SELECT 
        year,
        COUNT(*) as filas,
        COUNT(DISTINCT indicator_code) as indicadores_unicos,
        COUNT(DISTINCT pais) as paises
    FROM covid19.silver.mortalidad_costa_rica 
    GROUP BY year 
    ORDER BY year
""").show()

print("\n=== Resumen total ===")
spark.sql("""
    SELECT 
        COUNT(*) as total_filas,
        COUNT(DISTINCT year) as anos_cobertura,
        COUNT(DISTINCT indicator_code) as total_indicadores,
        MIN(year) as ano_min,
        MAX(year) as ano_max
    FROM covid19.silver.mortalidad_costa_rica
""").show()

In [0]:
spark.sql("""
    SELECT year, COUNT(*) as filas, COUNT(DISTINCT indicator_code) as indicadores
    FROM covid19.silver.mortalidad_costa_rica
    WHERE year >= 2020
    GROUP BY year
    ORDER BY year
""").show()

# Aqui Cargamos los codigos de enfermedades

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import requests
import io

# ==========================================
# PASO 1: Descargar CSV (saltando primera fila)
# ==========================================
dropbox_url = "https://www.dropbox.com/scl/fi/fr8p3jvqj9xckisiru5ot/enfermedades.csv?rlkey=cphzsdsi7cc6ll15c1qjxfuxn&st=aow8stul&dl=0"
download_url = dropbox_url.replace("&dl=0", "&dl=1")

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(download_url, headers=headers)
csv_content = response.content.decode('utf-8-sig')

# Leer CSV saltando la primera fila (skiprows=1) y usando la segunda como header
pdf = pd.read_csv(
    io.StringIO(csv_content),
    skiprows=1,  # Saltar la primera fila con nombres largos
    header=0     # Usar la siguiente fila como header (CAUSA, DESCRIP)
)

print(f"✅ CSV cargado: {len(pdf)} filas")
print(f"📋 Columnas: {pdf.columns.tolist()}")
display(pdf.head(5))

# Convertir a Spark DataFrame
df_bronze = spark.createDataFrame(pdf)

# Guardar en Bronze (esquema metadata)
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("covid19.metadata.codigos_cie10_enfermedades")

print("\n✓ Tabla Bronze guardada en covid19.metadata.codigos_cie10_enfermedades")

# ==========================================
# PASO 2: Transformar a SILVER con snake_case
# ==========================================
print("\n🔄 Transformando a Silver con snake_case...")

df_silver = (
    df_bronze
    # Renombrar columnas a minúsculas
    .withColumnRenamed("CAUSA", "causa")
    .withColumnRenamed("DESCRIP", "descripcion")
    # Convertir descripción a snake_case
    .withColumn(
        "descripcion_snake_case",
        F.lower(
            F.regexp_replace(
                F.regexp_replace(F.col("descripcion"), r"[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]", ""),  # Eliminar caracteres especiales (manteniendo acentos y ñ)
                r"\s+", "_"  # Reemplazar espacios con guión bajo
            )
        )
    )
    # Estandarizar código a mayúsculas
    .withColumn("causa", F.upper(F.col("causa")))
    # Limpiar descripción (quitar comillas si las hay)
    .withColumn("descripcion", F.trim(F.regexp_replace(F.col("descripcion"), r'^"|"$', "")))
    # Agregar metadatos
    .withColumn("pais_origen", F.lit("Guatemala/Centroamérica"))
    .withColumn("metadata_processed_at", F.current_timestamp())
    # Eliminar duplicados
    .dropDuplicates(["causa"])
)

print(f"✅ Silver creada: {df_silver.count()} filas")
print("\nVista previa de Silver:")
display(df_silver.select("causa", "descripcion", "descripcion_snake_case").limit(10))

# Guardar en Silver (esquema metadata)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("covid19.metadata.codigos_cie10_enfermedades_silver")

print("\n✓ Tabla Silver guardada en covid19.metadata.codigos_cie10_enfermedades_silver")

# ==========================================
# PASO 3: Verificar resultados
# ==========================================
print("\n" + "="*80)
print("VERIFICACIÓN FINAL")
print("="*80)

print("\n📊 Conteo de registros:")
spark.sql("SELECT COUNT(*) as total FROM covid19.metadata.codigos_cie10_enfermedades").show()
spark.sql("SELECT COUNT(*) as total FROM covid19.metadata.codigos_cie10_enfermedades_silver").show()

print("\n🔍 Ejemplos de transformación:")
spark.sql("""
    SELECT causa, descripcion, descripcion_snake_case 
    FROM covid19.metadata.codigos_cie10_enfermedades_silver 
    WHERE descripcion LIKE '%,%' OR descripcion LIKE '% %'
    LIMIT 10
""").show(truncate=False)

In [0]:
# Verificar nombres reales de columnas
print(" Columnas reales en el DataFrame Bronze:")
print(df_bronze.columns)
print("\n Schema completo:")
df_bronze.printSchema()
print("\n Primeras filas:")
display(df_bronze.limit(5))